# 02 — Cointegration analysis

Pairs trading rests on one statistical claim: that some linear combination of the two
prices is **stationary**, so that when the spread widens it can be expected to come back.
This notebook tests that claim for all 10 pairs — and then asks the question that actually
matters, which the textbook presentation almost never does:

> **Does cointegration measured on the in-sample window still hold out-of-sample?**

That distinction turns out to be the whole story. A pair that tests cointegrated on data
you have already seen tells you nothing if the property evaporates in the period you have to
trade through.

**A note on what is *not* happening here.** Nothing in this notebook selects pairs. All ten
were specified from economic reasoning and all ten are traded regardless of what their
p-values say. Filtering the universe down to whatever tests cointegrated would be textbook
data-snooping — selecting on a statistic computed from the same data used to evaluate the
result. These tests are **diagnostics that explain the backtest**, not a screen that
produces it.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pairs_teardown.config import load_config
from pairs_teardown.data.clean import align_prices, handle_missing, to_log_prices
from pairs_teardown.data.loaders import load_or_download
from pairs_teardown.signals.spread import build_rolling_spread
from pairs_teardown.stats.cointegration import (
    adf_pvalue,
    analyze_pair,
    build_spread,
    estimate_hedge_ratio,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "pairs.yaml")
WINDOW = cfg.signal.window

prices_raw = load_or_download(
    list(cfg.tickers), cfg.data.start, cfg.data.end, ROOT / cfg.data.cache_dir
)
pairs = {p.name: align_prices(handle_missing(prices_raw[[p.a, p.b]])) for p in cfg.pairs}
logs = {name: to_log_prices(df) for name, df in pairs.items()}

NAMES = [p.name for p in cfg.pairs]
print(f"{len(NAMES)} pairs | split at {cfg.split.in_sample_end} | window {WINDOW}")


## 1. Why log prices

Everything below uses **log** prices. On raw prices a spread between two assets that both
grow multiplicatively is heteroskedastic by construction — a \$2 gap between \$20 stocks
and a \$2 gap between \$200 stocks are not the same event, but a level spread scores them
identically. Taking logs turns proportional moves into equal-sized moves and is what makes
the constant-variance assumption behind a z-score defensible.

The difference is not cosmetic:


In [ ]:
rows = []
for p in cfg.pairs:
    df, lg = pairs[p.name], logs[p.name]
    raw_res = analyze_pair(pd.Series(df[p.a]), pd.Series(df[p.b]))
    log_res = analyze_pair(pd.Series(lg[p.a]), pd.Series(lg[p.b]))
    rows.append({
        "pair": p.name,
        "raw eg p": raw_res.eg_pvalue,
        "log eg p": log_res.eg_pvalue,
        "raw coint@5%": raw_res.eg_pvalue < 0.05,
        "log coint@5%": log_res.eg_pvalue < 0.05,
    })
levels = pd.DataFrame(rows).set_index("pair")
display(levels.round(4))
print(f"cointegrated on raw prices: {int(levels['raw coint@5%'].sum())}/10   "
      f"on log prices: {int(levels['log coint@5%'].sum())}/10")


The headline count is 2 of 10 either way — but that stability is a coincidence,
and the per-pair column shows it. **Two pairs swap verdicts:** SPY/VOO is not cointegrated
on raw prices (p = 0.49) and clearly is on logs (p = 0.008); UNP/CSX does the reverse
(p = 0.005 raw, p = 0.13 log).

That is worth sitting with. A specification choice made before any test is run — one that
most write-ups do not even mention — flips the answer for a fifth of the universe. Logs are
used from here on because the argument for them is methodological, not because they produce
more cointegration; as the count shows, they do not.


## 2. The central test: does cointegration persist?

Now the question that matters. For each pair, run the Engle–Granger and ADF tests
**separately** on the in-sample and out-of-sample windows, and fit a hedge ratio on each.

If the strategy's premise holds, a pair that is cointegrated in-sample should still be
cointegrated out-of-sample, and its hedge ratio should be roughly the same number in both
periods. A trader in December 2021 could only have known the in-sample column.


In [ ]:
rows = []
for p in cfg.pairs:
    lg = logs[p.name]
    is_m = cfg.split.is_mask(lg.index)
    oos_m = cfg.split.oos_mask(lg.index)

    r = {"pair": p.name}
    for tag, mask in [("IS", is_m), ("OOS", oos_m)]:
        res = analyze_pair(pd.Series(lg[p.a][mask]), pd.Series(lg[p.b][mask]))
        r[f"{tag} g"] = res.hedge_ratio
        r[f"{tag} eg p"] = res.eg_pvalue
        r[f"{tag} adf p"] = res.adf_pvalue
        r[f"{tag} joh"] = res.johansen.trace_stat
        r[f"{tag} joh 5%"] = res.johansen.cointegrated_5pct
        r[f"{tag} HL"] = res.half_life.half_life
        r[f"{tag} HL se"] = res.half_life.se
    JOH_CRIT = res.johansen.crit_95
    rows.append(r)

coint = pd.DataFrame(rows).set_index("pair")
coint["coint IS"] = coint["IS eg p"] < 0.05
coint["coint OOS"] = coint["OOS eg p"] < 0.05
coint["|Δg|"] = (coint["OOS g"] - coint["IS g"]).abs()
coint[["IS g", "OOS g", "|Δg|", "IS eg p", "OOS eg p", "coint IS", "coint OOS"]].round(3)


In [ ]:
both = coint["coint IS"] & coint["coint OOS"]
print(f"cointegrated at 5% in-sample:      {int(coint['coint IS'].sum())} of 10")
print(f"cointegrated at 5% out-of-sample:  {int(coint['coint OOS'].sum())} of 10")
print(f"cointegrated in BOTH periods:      {int(both.sum())} of 10   -> {sorted(coint[both].index)}")
print(f"\nlost it  IS -> OOS: {sorted(coint[coint['coint IS'] & ~coint['coint OOS']].index)}")
print(f"gained it IS -> OOS: {sorted(coint[~coint['coint IS'] & coint['coint OOS']].index)}")


**Three pairs test cointegrated in-sample. Three test cointegrated out-of-sample.
Only one is the same pair in both.**

MA/V is the sole pair that holds the property across the split. WM/RSG and KO/PEP have it
and lose it; SPY/VOO and UNP/CSX lack it and acquire it. The count is stable at three
because pairs are swapping in and out, not because three pairs are reliably cointegrated.

This is the study's most important diagnostic, and it undercuts the strategy's premise
directly:

> **Cointegration, as measured on the window you can see, is close to uninformative about
> the window you have to trade.** A pair passing the test in December 2021 was about as
> likely to fail it over 2022–2024 as to pass.

It also explains, mechanically, the enormous cross-sectional dispersion reported in
`03_backtest_results.ipynb`. The strategy is not being applied to a set of stably
mean-reverting spreads with varying edge sizes. It is being applied to spreads whose
mean-reverting character switches on and off, and whether a given pair happened to be
reverting during 2022–2024 is largely luck.


### 2b. The same question, asked a second way: Johansen

Engle–Granger is a *residual-based* test: regress A on B, then ask whether the residual has
a unit root. Two things about that construction are worth knowing before trusting a count.

**It is asymmetric.** The OLS slope of A on B is not the reciprocal of the slope of B on A,
so the test can give a different p-value depending on which leg is named first. The study
fixes the direction in `configs/pairs.yaml` before any test is run, which makes it a
convention rather than a degree of freedom — but it is still a convention.

**Its residual is the most stationary-looking combination available.** OLS chooses the
hedge ratio that minimises residual variance, and a small-variance residual is also one
that looks like it reverts. Testing that residual with an ordinary ADF table therefore
rejects the unit root too often; this is the point Engle and Granger (1987) made, and
Phillips and Ouliaris (1990) tabulated the corrected critical values that `coint` uses.
The `adf p` column above uses the *uncorrected* table, which is why it is smaller than
`eg p` for every pair and why it is not the number this notebook counts.

The **Johansen trace test** avoids both problems. It treats (A, B) as a system, estimates
the rank of the long-run relationship directly, and is invariant to which series is listed
first. It reports a statistic against tabulated critical values rather than a p-value.

In [ ]:
joh = coint[["IS eg p", "IS joh", "IS joh 5%", "OOS eg p", "OOS joh", "OOS joh 5%"]].copy()
joh["EG IS"] = coint["coint IS"]
joh["EG OOS"] = coint["coint OOS"]
print(f"Johansen trace statistic, 5% critical value = {JOH_CRIT:.2f}\n")
display(joh[["IS eg p", "EG IS", "IS joh", "IS joh 5%",
             "OOS eg p", "EG OOS", "OOS joh", "OOS joh 5%"]].round(3))

for tag in ["IS", "OOS"]:
    eg = set(coint.index[coint[f"coint {tag}"]])
    jo = set(coint.index[coint[f"{tag} joh 5%"]])
    print(f"{tag:>3}: Engle-Granger {sorted(eg)}")
    print(f"     Johansen      {sorted(jo)}")
    print(f"     agree on      {sorted(eg & jo)}\n")
joh_both = coint["IS joh 5%"] & coint["OOS joh 5%"]
print(f"Johansen: cointegrated in BOTH periods: {sorted(coint.index[joh_both])}")

**The two tests disagree on most of the pairs they flag.** In-sample, Engle–Granger
selects WM/RSG, KO/PEP and MA/V; Johansen selects SPY/VOO and MA/V. They agree on one pair.
Out-of-sample Johansen flags five pairs to Engle–Granger's three, again overlapping on MA/V
and SPY/VOO only.

SPY/VOO is the instructive case. Two ETFs on the same index are cointegrated by
construction, and Johansen says so with a trace statistic eight times its critical value.
Engle–Granger returns p = 0.09 in-sample — not significant at 5%. The reason is mechanical:
the spread reverts in about a day (§3b below), so its residual is nearly white noise, and
the augmented Dickey–Fuller regression underneath `coint` has almost nothing to fit; the
test loses power on exactly the pair where cointegration is most obvious. A binary count
at 5% turns that quirk into "not cointegrated".

What survives both tests in both periods is still one pair — MA/V — with SPY/VOO added if
Johansen is the arbiter. The persistence conclusion of §2 does not depend on the test. The
lesson is narrower and worth stating: **a cointegration count is a property of the test as
much as of the pairs**, and a study that reports one test's count as "the number of
cointegrated pairs" is reporting a specification choice as a fact.

## 3. Hedge ratio instability

Even setting the tests aside: is the *hedge ratio* stable? The strategy fits one number
in-sample and applies it unchanged for three years. If that number would have been very
different had it been fitted on the out-of-sample data, the position sizing is being driven
by a stale estimate.


In [ ]:
drift = coint[["IS g", "OOS g", "|Δg|"]].copy()
drift["sign flip"] = np.sign(coint["IS g"]) != np.sign(coint["OOS g"])
drift.sort_values("|Δg|", ascending=False).round(3)


**UPS/FDX's hedge ratio does not merely drift — it changes sign**, from +0.84
in-sample to −0.35 out-of-sample.

A sign flip is qualitatively different from drift. A positive ratio means "long A, short B";
a negative one means the fitted relationship says *long both*. The strategy holds the
in-sample +0.84 throughout, so for the whole out-of-sample period it is hedging in the
opposite direction to the relationship the data actually exhibits. UPS/FDX is the study's
worst performer at **−42.4%** net out-of-sample, and this is why.

XOM/CVX is the next worst offender (0.46 → 1.30, nearly 3x) and is the study's
second-worst performer at −26.1%. The ordering is not a coincidence.

Note the deliberate design choice this exposes and does *not* fix: the sizing ratio stays
static precisely because a rolling estimate is dangerous in the other direction — a noisy
rolling ratio that wanders toward zero destroys market neutrality entirely (see
`05_writeup.ipynb` §2.1 for the +521% vs −61.6% synthetic test). **Both estimators fail on
UPS/FDX; the static one merely fails more predictably.** The honest conclusion is that no
hedge-ratio specification rescues a pair whose economic relationship has genuinely
changed.


## 3b. How fast does the spread revert? The half-life

A p-value says whether a spread reverts. The strategy needs to know *how fast*, because the
z-score window (60 days) and the cost per round trip both only make sense on a particular
time scale. The standard way to put a number on this is to fit the spread as a discrete
Ornstein–Uhlenbeck process:

$$\Delta s_t = \alpha + \varphi\, s_{t-1} + \varepsilon_t, \qquad
\text{half-life} = \frac{-\ln 2}{\ln(1 + \varphi)}.$$

This is the AR(1) model $s_t = c + \rho s_{t-1} + \varepsilon_t$ with $\rho = 1 + \varphi$,
and $\varphi$ is the same coefficient the Dickey–Fuller test asks "is it zero?" about —
here we ask "how big is it?" instead. `stats.cointegration.half_life` returns the estimate
with a **delta-method standard error**, $\text{SE}(\text{HL}) = \left|\frac{d\,\text{HL}}{d\varphi}\right| \text{SE}(\varphi)$,
and the SE turns out to matter as much as the estimate.

Two columns below. `IS HL` is fitted entirely in-sample. `OOS HL of IS spread` takes the
spread a trader would actually have held — the in-sample hedge ratio applied to
out-of-sample prices — and measures how fast *that* reverted in 2022–2024. (Fitting a fresh
hedge ratio on the out-of-sample window and then measuring its half-life would be
answering a question nobody could have asked in December 2021.)

In [ ]:
from pairs_teardown.stats.cointegration import half_life

rows = []
for p in cfg.pairs:
    lg = logs[p.name]
    is_m, oos_m = cfg.split.is_mask(lg.index), cfg.split.oos_mask(lg.index)
    la, lb = pd.Series(lg[p.a]), pd.Series(lg[p.b])
    g_is = estimate_hedge_ratio(la[is_m], lb[is_m])
    held = build_spread(la, lb, g_is)          # the spread the strategy actually sizes on
    hl_is = half_life(held[is_m])
    hl_oos = half_life(held[oos_m])
    rows.append({
        "pair": p.name,
        "IS HL (days)": hl_is.half_life, "IS se": hl_is.se,
        "IS phi": hl_is.phi, "IS phi se": hl_is.phi_se,
        "OOS HL of IS spread": hl_oos.half_life, "OOS se": hl_oos.se,
    })

hl = pd.DataFrame(rows).set_index("pair")
hl["HL > window (IS)"] = hl["IS HL (days)"] > WINDOW
# phi within two SEs of zero: the unit root is not rejected, so the half-life
# cannot be told apart from infinite whatever its point estimate says.
hl["unit root not rejected (IS)"] = hl["IS phi"] + 2 * hl["IS phi se"] >= 0
display(hl.sort_values("IS HL (days)").round(3))

print(f"z-score window: {WINDOW} days")
print(f"in-sample half-life longer than the window:   {int(hl['HL > window (IS)'].sum())} of 10")
print(f"in-sample phi within 2 SE of zero (no reversion detectable): "
      f"{int(hl['unit root not rejected (IS)'].sum())} of 10   "
      f"-> {sorted(hl.index[hl['unit root not rejected (IS)']])}")

Three things in this table, in decreasing order of comfort.

**The half-lives are longer than the window for half the universe.** In-sample, five
pairs revert in under 60 days (SPY/VOO in about one day, MA/V in ~32, KO/PEP, FOXA/FOX and
WM/RSG in the 40s) and five take longer — DUK/SO ~77, HD/LOW and UNP/CSX ~95, XOM/CVX and
UPS/FDX in the hundreds. A rolling z-score with a 60-day window compares today's spread
with its own trailing 60-day mean. When the half-life is 95 days, that trailing mean has
not settled at the equilibrium; it is following the spread's slow wander. What the signal
then trades is short-run noise around a drifting local level, not reversion to a long-run
relationship. This is also why the *rolling* spread in §4 passes ADF for 10 of 10 pairs:
subtract a local level from anything and what remains looks stationary.

**For the two slowest pairs, no reversion is detectable at all.** XOM/CVX (299 ± 200 days)
and UPS/FDX (522 ± 688) have φ within two standard errors of zero: the unit root is not
rejected, and the half-life cannot be told apart from infinite. These are not slow
reversion, they are "indistinguishable from a random walk". The point estimate of a
half-life is *always* finite on a finite sample — OLS on a unit-root series is biased
toward reversion (the Dickey–Fuller bias) — so a half-life without its SE is a number that
looks like information and is not. This is the reason `half_life` refuses to return the
estimate on its own.

**Out-of-sample reversion speed does not sort the winners from the losers.** UNP/CSX and
HD/LOW, the two best performers, do speed up (95 → 35 and 94 → 36 days). But KO/PEP made
+14% while its held spread became unmeasurable (175 ± 172), and MA/V reverted fastest of
any real pair (23 ± 7) and made nothing. Half-life explains the *mechanism* of the window
mismatch; it does not predict which pairs will pay, and it should not be used to pick them.

**What this does not license.** Matching the window to each pair's half-life would be
the natural fix, and it is exactly the parameter search the study forbids: the half-lives
are estimated from the same data the strategy is judged on. The finding stands as a
diagnosis — the pre-registered 60 was too short for half the universe — not as a
prescription.

## 4. Static versus rolling spreads

This is the decision recorded as `signal_hedge: rolling` in the config, and §3 is the
reason for it. A static hedge ratio produces a spread that inherits every bit of the
relationship's drift; a rolling one re-estimates and absorbs it.

Testing stationarity of both spread constructions on the in-sample window — the only window
that choice could legitimately have been made from:


In [ ]:
rows = []
for p in cfg.pairs:
    lg = logs[p.name]
    is_m = cfg.split.is_mask(lg.index)
    la, lb = pd.Series(lg[p.a]), pd.Series(lg[p.b])

    g_is = estimate_hedge_ratio(la[is_m], lb[is_m])
    sp_static = build_spread(la, lb, g_is)
    sp_rolling = build_rolling_spread(la, lb, WINDOW)

    rows.append({
        "pair": p.name,
        "static adf p": adf_pvalue(sp_static[is_m].dropna()),
        "rolling adf p": adf_pvalue(sp_rolling[is_m].dropna()),
    })

spec = pd.DataFrame(rows).set_index("pair")
spec["static stationary@5%"] = spec["static adf p"] < 0.05
spec["rolling stationary@5%"] = spec["rolling adf p"] < 0.05
display(spec.round(4))
print(f"in-sample stationary spread — static: {int(spec['static stationary@5%'].sum())}/10   "
      f"rolling: {int(spec['rolling stationary@5%'].sum())}/10")


**Static: 4 of 10 stationary. Rolling: 10 of 10.** That is the ADF justification
the config comment refers to, and on its face it looks decisive.

It should be read with suspicion rather than as a free win, and the perfect score is exactly
why. A rolling hedge ratio re-estimated every 60 days will *tend* to produce something that
looks stationary almost mechanically — it is continuously refitting the very level it then
measures deviations from. A construction that passes at 10 out of 10, including for UPS/FDX
(static p = 0.82) whose two legs genuinely came apart over this period, is telling you more
about the estimator than about the pairs. Passing an ADF test on such a series is a much
weaker statement than passing one on a fixed linear combination.

Whether it actually helps the strategy is an empirical question, not a statistical one, and
it is answered in `04_sensitivity_analysis.ipynb` §4 by running the backtest both ways. The
short version: **it makes almost no difference to returns** (mean out-of-sample +0.64% with
rolling versus −0.38% with static), which is a useful corrective to how consequential this
choice looks here.


## 5. What the spreads actually look like

The rolling spread and its z-score for every pair, with the ±2.0 entry and ±0.5 exit bands
marked. The in-sample/out-of-sample boundary is dashed.


In [ ]:
fig, axes = plt.subplots(len(cfg.pairs), 1, figsize=(12, 2.6 * len(cfg.pairs)),
                         sharex=False)
split = pd.Timestamp(cfg.split.in_sample_end)

for ax, p in zip(axes, cfg.pairs):
    lg = logs[p.name]
    spread = build_rolling_spread(pd.Series(lg[p.a]), pd.Series(lg[p.b]), WINDOW)
    z = (spread - spread.rolling(WINDOW).mean()) / spread.rolling(WINDOW).std()

    ax.plot(z.index, z, lw=0.7, color="steelblue")
    for lvl, ls in [(cfg.signal.entry, "--"), (-cfg.signal.entry, "--"),
                    (cfg.signal.exit, ":"), (-cfg.signal.exit, ":")]:
        ax.axhline(lvl, color="firebrick", ls=ls, lw=0.8)
    ax.axvline(split, color="0.4", ls="--", lw=1)
    ax.set_title(f"{p.name} — rolling z-score (window {WINDOW})", fontsize=10, loc="left")
    ax.set_ylim(-5, 5)

plt.tight_layout()
plt.show()


Every pair's z-score crosses the ±2 bands repeatedly in both periods, so every pair
trades actively throughout — the differences in outcome reported in
`03_backtest_results.ipynb` are not caused by some pairs sitting idle.

What the eye cannot see here, and what §2 established, is that a crossing followed by
reversion and a crossing followed by continued divergence look identical at the moment of
entry. That is the strategy's actual problem.

**Next:** `03_backtest_results.ipynb` runs the backtest and reports what these spreads were
worth.
